# Vision Metrics

Evaluate a Vision Language Model (VLM) using two metrics based on semantic similarity between the model's description and the human ground truth.

| Metric | What it tells you |
|--------|-------------------|
| **VisionSimilarity** | How semantically close the VLM descriptions are to the ground truth on average. A score of 1.0 means identical meaning; 0.0 means completely unrelated. |
| **VisionHallucination** | How often the VLM describes something significantly different from what actually happened. A frame is a hallucination when similarity falls below `threshold`. |

In [ ]:
!pip install alquimia-fair-forge

Defaulting to user installation because normal site-packages is not writeable
ERROR: Ignored the following versions that require a different python version: 1.0.0 Requires-Python >=3.11; 1.1.0 Requires-Python >=3.11; 1.2.0 Requires-Python >=3.11; 1.2.0b1 Requires-Python >=3.11; 1.2.0b2 Requires-Python >=3.11; 1.2.0b3 Requires-Python >=3.11; 1.2.0b4 Requires-Python >=3.11; 1.2.0b5 Requires-Python >=3.11; 1.2.0b6 Requires-Python >=3.11; 1.2.0b7 Requires-Python >=3.11; 1.3.0b1 Requires-Python >=3.11; 2.0.0 Requires-Python >=3.11; 2.0.0b1 Requires-Python >=3.11; 2.0.0b2 Requires-Python >=3.11; 2.0.0b3 Requires-Python >=3.11; 3.0.0b1 Requires-Python >=3.11
ERROR: Could not find a version that satisfies the requirement alquimia-fair-forge (from versions: none)

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip
ERROR: No matching distribution found for alquimia-fair-forge


In [1]:
import logging
from fair_forge.metrics.vision import VisionSimilarity, VisionHallucination

logging.getLogger('httpx').disabled = True

## Setup

Implement a `Retriever` that loads your evaluation dataset.

Each `Batch` only needs two text fields:
- `assistant` — free-text description produced by the VLM
- `ground_truth_assistant` — human-annotated description of what actually happened

In [ ]:
import json
from pathlib import Path
from fair_forge import Retriever
from fair_forge.schemas import Dataset


class VLMRetriever(Retriever):
    def load_dataset(self) -> list[Dataset]:
        with open(Path('dataset.json'), encoding='utf-8') as f:
            return [Dataset.model_validate(item) for item in json.load(f)]


# Threshold — frames with similarity below this value are flagged as hallucinations.
# Adjust based on your domain:
#   0.85+ → strict (only near-identical descriptions pass)
#   0.75  → balanced (default)
#   0.60  → lenient (allows paraphrasing and partial descriptions)
THRESHOLD = 0.75

## Vision Similarity

How accurately does the VLM describe scenes compared to the human ground truth?

In [ ]:
similarity_results = VisionSimilarity.run(VLMRetriever, threshold=THRESHOLD)

for r in similarity_results:
    r.display()

## Vision Hallucination

How often does the VLM describe something significantly different from what actually happened?

In [ ]:
hallucination_results = VisionHallucination.run(VLMRetriever, threshold=THRESHOLD)

for r in hallucination_results:
    r.display()